# Основа для ЧП3: сплит, признаки, метрики, бейзлайн

Общая точка старта для KNN, линейных моделей и экспериментов с признаками. Описание в `ml/README.md`.

Данные приватные: строки таблиц в выводах не показываем, только размеры, даты и метрики.

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from dayahead.metrics import evaluate
from dayahead.split import load_splits

s = load_splits()

## Сплит

По времени, без перемешивания: случайный сплит дал бы утечку из будущего через лаги цены.

In [2]:
parts = {"train": s.y_train, "validation": s.y_val, "test": s.y_test}
pd.DataFrame({
    "с": {k: y.index.min().date() for k, y in parts.items()},
    "по": {k: y.index.max().date() for k, y in parts.items()},
    "часов": {k: len(y) for k, y in parts.items()},
    "цена <= 0": {k: int((y <= 0).sum()) for k, y in parts.items()},
    "цена > 300": {k: int((y > 300).sum()) for k, y in parts.items()},
})

,с,по,часов,цена <= 0,цена > 300
train,2024-01-08,2024-12-31,8593,305,222
validation,2025-01-01,2025-06-30,4343,231,43
test,2025-07-01,2025-12-31,4417,113,46


Train начинается не с 1 января: первые строки без недельных лагов отброшены.

## Признаки

In [3]:
print(len(s.X_train.columns), "признаков:")
print(*s.X_train.columns, sep="\n")

17 признаков:
demand_forecast_mwh
solar_forecast_mwh
net_exchange_forecast_mwh
wind_forecast_mwh
nuclear_forecast_mwh
temperature_forecast_c
hour
dow
is_weekend
is_holiday
residual_load
price_lag_24
price_lag_48
price_lag_168
price_roll_mean_24
price_roll_std_24
price_roll_mean_168


## Бейзлайн

Цена вчера в этот час (`price_lag_24`) и неделю назад (`price_lag_168`). Ничего не обучается, поэтому test здесь смотреть можно.

In [4]:
rows = {}
for name, X, y in [("validation", s.X_val, s.y_val), ("test", s.X_test, s.y_test)]:
    rows[("naive-24", name)] = evaluate(y, X["price_lag_24"])
    rows[("naive-168", name)] = evaluate(y, X["price_lag_168"])
pd.DataFrame(rows).T.round(2)

,,MAE,RMSE,MAE_le0,MAE_gt300
naive-24,validation,26.82,42.02,35.36,135.41
naive-168,validation,31.22,46.61,39.18,152.79
naive-24,test,23.03,36.41,25.51,108.77
naive-168,test,25.56,41.23,24.08,161.15


## Пример модели

Шаблон для своих экспериментов: предобработка внутри `Pipeline`, обучение на train, оценка на validation. Модель без подбора, только чтобы показать порядок работы.

In [5]:
CAT = ["hour", "dow"]
NUM = [c for c in s.X_train.columns if c not in CAT]

pipe = Pipeline([
    ("pre", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
        ("num", StandardScaler(), NUM),
    ])),
    ("model", LinearRegression()),
])
pipe.fit(s.X_train, s.y_train)

pd.DataFrame({
    "naive-24": evaluate(s.y_val, s.X_val["price_lag_24"]),
    "LinearRegression": evaluate(s.y_val, pipe.predict(s.X_val)),
}).T.round(2)

,MAE,RMSE,MAE_le0,MAE_gt300
naive-24,26.82,42.02,35.36,135.41
LinearRegression,22.06,31.82,25.84,131.26


Месяца в базовом наборе нет намеренно: train покрывает один год, и one-hot месяца запоминает уровень цен 2024. С ним эта же модель на validation хуже бейзлайна (MAE 30.3 против 26.8).